# SignScribe: Arabic (ArSL) letter models from ArSL21L

Trains, evaluates and exports the two files the web app loads for Arabic letters:
`letters-arsl-mlp.json` (the app's first choice) and `letters-arsl-knn.json` (fallback).
It runs the repo's own scripts (`ml/import_images.py`, `ml/evaluate.py`, `ml/calibrate.py`, `ml/export_arsl.py`,
`ml/validate_model.py`) at commit `c88ee4e`, so the feature vector (`hand60-v1`) and the JSON format are exactly what the app expects.

**Dataset:** ArSL21L, Arabic Sign Language Letter Dataset (Gochoo et al., UAE University), CC BY 4.0,
https://data.mendeley.com/datasets/8hrn3bvdvk/1 (an earlier record is f63xhm286w). 14,202 pictures, 32 classes, 50 people.

**Kaggle settings (right panel):** Internet **On** (git clone, pip, the hand model); Accelerator **None** (CPU is enough).

**Getting the data:** best is to upload the ArSL21L zip you already downloaded as a **private** Kaggle dataset and attach it with
*+ Add Input*. If nothing is attached, the notebook tries Mendeley's public API (not tested from where this notebook was written:
if it fails, the cell tells you what to do).

**Slow step:** step 4 runs MediaPipe on every picture.

## Before you share anything from this notebook
- **Keep this notebook and its output private.** The `*-knn.json` file stores real hand landmark samples
  (the repo never commits it), and the takes file holds landmarks of every picture.
- **Only quote numbers that a script wrote** (`MODEL_RESULTS.md`, the `reports/` files). These are numbers on **dataset pictures**,
  not on a webcam: real-camera accuracy is still **NOT MEASURED**.
- `MODEL_RESULTS.md` in the output is the repo's file with this run appended by `ml/evaluate.py`; it replaces `docs/MODEL_RESULTS.md`
  in a branch (then update section 6 of `docs/PROJECT_STATUS.md`; `npm run check` compares the two).


## 1. Settings

In [ ]:
REPO_URL = "https://github.com/Youssef-Adel91/SignScribe.git"
REPO_COMMIT = "c88ee4e"      # main on 2026-11-01; set to None for the newest main
NOTEBOOK_NAME = "signscribe_arsl_letters"

INPUT_PATH = None                  # e.g. "/kaggle/input/arsl21l"; None = use whatever is attached, else download
MENDELEY_ID, MENDELEY_VERSION = "8hrn3bvdvk", 1
DATASET_NAME = "ArSL21L"
DATASET_VERSION = "Mendeley 8hrn3bvdvk v1, as published"   # write the record you really used

# ArSL21L file names: <person>_<age>_<sex>_<class>_<n>.jpg (the same regexes as ml/README.md)
LABEL_REGEX = r"_([a-z]+)_\d+\.jpg$"
PERSON_REGEX = r"^(\d+)_"
AGE_REGEX = r"^\d+_(\d+)_"       # only used to mark people under 18; the age is never stored

REIMPORT = False                   # True = run MediaPipe again even if the takes file exists
KEEP_TAKES = False                 # True = also put the takes file (landmarks of every picture) in the output

## 2. Setup

In [ ]:
# ---- setup: paths, a helper that runs a command and streams its output ----
import hashlib, json, os, random, re, shutil, subprocess, sys, time, urllib.request, zipfile
from pathlib import Path

IN_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if IN_KAGGLE else Path.cwd() / "work"
SCRATCH = Path("/kaggle/temp") if Path("/kaggle/temp").exists() else Path("/tmp/signscribe")
WORK.mkdir(parents=True, exist_ok=True)
SCRATCH.mkdir(parents=True, exist_ok=True)

# only for checking this notebook itself with generated hands: never set it for a real run
PIPELINE_TEST = os.environ.get("SIGNSCRIBE_PIPELINE_TEST") == "1"
if PIPELINE_TEST:
    print("PIPELINE TEST MODE: generated data, the results mean nothing")

def run(cmd, cwd=None):
    """Runs a command, prints its output while it runs, stops the notebook if it fails."""
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"command failed with exit code {p.returncode}: {' '.join(cmd[:3])} ...")

def sha256_of(path, n=16):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()[:n]

print("work folder (kept as notebook output):", WORK)
print("scratch folder (not kept):", SCRATCH)

In [ ]:
# ---- get the SignScribe code (the same ml/ scripts the team runs) + MediaPipe + the hand model ----
REPO = Path(os.environ.get("SIGNSCRIBE_REPO_DIR", SCRATCH / "SignScribe"))
if not (REPO / ".git").exists():
    run(["git", "clone", "--quiet", REPO_URL, REPO])
if REPO_COMMIT and not os.environ.get("SIGNSCRIBE_REPO_DIR"):
    run(["git", "checkout", "--quiet", REPO_COMMIT], cwd=REPO)
COMMIT = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=REPO, text=True).strip()
print("SignScribe commit:", COMMIT)

try:
    import mediapipe  # noqa: F401
except ImportError:
    run([sys.executable, "-m", "pip", "install", "--quiet", "mediapipe>=0.10.14,<0.11", "numpy"])
import mediapipe
print("mediapipe", mediapipe.__version__)

# the same file and URL as tools/get-models.mjs (npm run models)
HAND_MODEL = REPO / "public" / "models" / "hand_landmarker.task"
HAND_MODEL_URL = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
if not HAND_MODEL.exists() and not PIPELINE_TEST:
    HAND_MODEL.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(HAND_MODEL_URL, HAND_MODEL)
if HAND_MODEL.exists():
    print("hand model:", HAND_MODEL.name, f"{HAND_MODEL.stat().st_size/1e6:.1f} MB, sha256", sha256_of(HAND_MODEL))

In [ ]:
# ---- dataset helpers: use an attached Kaggle dataset, else download from Mendeley Data ----
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def extract_all_zips(folder):
    """Extracts every .zip under `folder` next to itself (zips inside zips too), then deletes the extracted zip copy."""
    done = set()
    while True:
        zips = [z for z in Path(folder).rglob("*.zip") if z not in done]
        if not zips:
            return
        for z in zips:
            target = z.with_suffix("")
            print("extracting", z.relative_to(folder), flush=True)
            with zipfile.ZipFile(z) as zf:
                zf.extractall(target)
            done.add(z)
            z.unlink()  # only copies in the scratch folder are ever deleted

def mendeley_download(dataset_id, version, dest):
    """Downloads every file of a public Mendeley Data dataset through its public API."""
    dest = Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    def get_json(url):
        req = urllib.request.Request(url, headers={"Accept": "application/json", "User-Agent": "signscribe-notebook"})
        with urllib.request.urlopen(req, timeout=60) as r:
            return json.load(r)
    base = f"https://data.mendeley.com/public-api/datasets/{dataset_id}"
    folder_ids = ["root"]
    try:
        folder_ids += [f["id"] for f in get_json(f"{base}/folders/{version}")]
    except Exception as e:
        print("(no folder list:", e, ")")
    got = 0
    for fid in folder_ids:
        for f in get_json(f"{base}/files?folder_id={fid}&version={version}"):
            url = (f.get("content_details") or {}).get("download_url")
            if not url:
                continue
            out = dest / f["filename"]
            size = (f.get("content_details") or {}).get("size") or f.get("size")
            print(f"downloading {f['filename']} ({(size or 0)/1e6:.0f} MB)", flush=True)
            urllib.request.urlretrieve(url, out)
            got += 1
    if not got:
        raise RuntimeError("the Mendeley API listed no files")

def get_dataset(dest, input_path, dataset_id, version, page):
    """Returns a scratch folder with the dataset's files, zips extracted. Never writes into /kaggle/input."""
    dest = Path(dest)
    marker = dest / ".ready"
    if marker.exists():
        print("dataset already prepared in", dest)
        return dest
    dest.mkdir(parents=True, exist_ok=True)
    src = Path(input_path) if input_path else None
    if src is None and Path("/kaggle/input").exists() and any(Path("/kaggle/input").iterdir()):
        src = Path("/kaggle/input")
    if src is not None:
        print("using the attached files in", src)
        for z in src.rglob("*.zip"):  # zips are copied so they can be extracted; pictures are linked, not copied
            rel = z.relative_to(src)
            (dest / rel).parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(z, dest / rel)
        link = dest / "attached"
        if not link.exists():
            link.symlink_to(src.resolve(), target_is_directory=True)
    else:
        try:
            mendeley_download(dataset_id, version, dest)
        except Exception as e:
            raise RuntimeError(
                f"Automatic download failed ({e}).\n"
                f"Download the dataset in your browser from {page} , upload the zip as a PRIVATE Kaggle dataset, "
                f"attach it with '+ Add Input' (right panel) and run this cell again. Or set INPUT_PATH."
            ) from e
    extract_all_zips(dest)
    marker.write_text("ok")
    return dest

def images_under(folder):
    """Every picture under `folder`, following the link to an attached dataset."""
    found = []
    for d, _, files in os.walk(folder, followlinks=True):
        found += [Path(d) / f for f in files if Path(f).suffix.lower() in IMAGE_SUFFIXES]
    return sorted(found)

## 3. Get the dataset and look at it before importing

In [ ]:
RAW = get_dataset(SCRATCH / "arsl21l_raw", INPUT_PATH, MENDELEY_ID, MENDELEY_VERSION,
                  "https://data.mendeley.com/datasets/8hrn3bvdvk/1")

# one flat folder of links, one per unique file name (a file name is one picture: person, class and number are in it),
# so a copy of the pictures in two sub-folders (for example train/ and val/) is never imported twice
lrx = re.compile(LABEL_REGEX)
STAGE = SCRATCH / "arsl21l_stage"
if STAGE.exists():
    shutil.rmtree(STAGE)
STAGE.mkdir()
seen, total, unmatched = {}, 0, 0
for p in images_under(RAW):
    total += 1
    if not lrx.search(p.name):
        unmatched += 1
        continue
    if p.name not in seen:
        seen[p.name] = p
        (STAGE / p.name).symlink_to(p.resolve())
print(f"pictures found: {total}, file name not like <person>_<age>_<sex>_<class>_<n>.jpg: {unmatched}, "
      f"duplicate file names skipped: {total - unmatched - len(seen)}, staged: {len(seen)}")
classes = {}
for name in seen:
    classes[lrx.search(name).group(1)] = classes.get(lrx.search(name).group(1), 0) + 1
print(f"classes: {len(classes)} (expected 32)")
print("  ".join(f"{k}:{v}" for k, v in sorted(classes.items())))
people = {re.search(PERSON_REGEX, n).group(1) for n in seen if re.search(PERSON_REGEX, n)}
print("people found:", len(people), "(expected 50; only counted, never listed)")
assert len(seen) > 0, "no ArSL21L pictures found: check INPUT_PATH / the attached dataset"

## 4. Pictures → hand landmarks (MediaPipe) → takes file
Read the import report it prints: pictures with no hand are skipped and counted.

In [ ]:
TAKES = SCRATCH / "arsl21l.jsonl"
if REIMPORT or not TAKES.exists():
    run([sys.executable, REPO / "ml/import_images.py", "--images", STAGE, "--out", TAKES, "--overwrite",
         "--alphabet", "arsl", "--dataset-name", DATASET_NAME, "--source", "arsl21l",
         "--label-regex", LABEL_REGEX, "--person-regex", PERSON_REGEX, "--age-regex", AGE_REGEX,
         "--model", HAND_MODEL], cwd=REPO)
else:
    print("using the existing takes file", TAKES, "(set REIMPORT = True to import again)")
print("takes:", sum(1 for _ in open(TAKES)), " sha256", sha256_of(TAKES))

## 5. Honest evaluation: leave-one-person-out (each of the 50 people held out in turn)
With and without augmentation. The run is appended to the repo's `docs/MODEL_RESULTS.md` by the script.

In [ ]:
OUT = WORK / "signscribe-arsl-output"
REPORTS, MODELS = OUT / "reports", OUT / "models"
for d in (REPORTS, MODELS):
    d.mkdir(parents=True, exist_ok=True)
test_flag = ["--allow-synthetic"] if PIPELINE_TEST else []
results_md = [] if PIPELINE_TEST else ["--results-md", REPO / "docs/MODEL_RESULTS.md"]
before = set(REPORTS.glob("*.json"))
run([sys.executable, REPO / "ml/evaluate.py", "--data", TAKES, "--alphabet", "arsl",
     "--dataset-name", DATASET_NAME, "--dataset-version", DATASET_VERSION,
     "--augment", "all", "--augment-ablation", "--out-dir", REPORTS, *results_md, *test_flag], cwd=REPO)
EVAL_JSON = max(set(REPORTS.glob("*.json")) - before, key=lambda p: p.stat().st_mtime)
print("evaluation report:", EVAL_JSON.with_suffix(".md").name)

## 6. Confidence thresholds (a table to choose from by hand; it changes nothing)

In [ ]:
run([sys.executable, REPO / "ml/calibrate.py", "--data", TAKES, "--alphabet", "arsl",
     "--dataset-name", DATASET_NAME, "--dataset-version", DATASET_VERSION, "--out-dir", REPORTS, *test_flag], cwd=REPO)

## 7. Export the two model files (trained on all 50 people) and validate them

In [ ]:
run([sys.executable, REPO / "ml/export_arsl.py", "--data", TAKES, "--out-dir", MODELS,
     "--augment", "all", "--seed", "0", "--evaluation-json", EVAL_JSON, *test_flag], cwd=REPO)
for f in sorted(MODELS.glob("letters-arsl-*.json")):
    run([sys.executable, REPO / "ml/validate_model.py", f, "--alphabet", "arsl"], cwd=REPO)

## 8. Package the output

In [ ]:
# ---- collect everything the team needs into one folder and one zip (the notebook's output) ----
for p in [REPO / "docs" / "MODEL_RESULTS.md", TAKES.with_suffix(".import.md")]:
    if p.exists():
        shutil.copy2(p, OUT / ("MODEL_RESULTS.md" if p.name == "MODEL_RESULTS.md" else "reports/" + p.name))
if KEEP_TAKES and TAKES.exists():
    shutil.copy2(TAKES, OUT / TAKES.name)
info = {
    "notebook": NOTEBOOK_NAME,
    "signscribe_commit": COMMIT,
    "mediapipe": mediapipe.__version__,
    "hand_model_sha256_16": sha256_of(HAND_MODEL) if HAND_MODEL.exists() else None,
    "dataset": DATASET_NAME, "dataset_version": DATASET_VERSION,
    "takes_sha256_16": sha256_of(TAKES),
    "finished_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "pipeline_test": PIPELINE_TEST,
}
(OUT / "RUN_INFO.json").write_text(json.dumps(info, indent=1))
archive = shutil.make_archive(str(WORK / OUT.name), "zip", OUT)
print("\nOutput folder:", OUT)
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(OUT)}  ({p.stat().st_size/1e3:.0f} kB)")
print("zip:", archive)

## 9. Putting the models into the app (on your computer, in a branch)
1. Download the output zip (right panel → *Output*).
2. Copy `models/letters-arsl-mlp.json` and `models/letters-arsl-knn.json` into `public/models/`.
   **Never commit the k-NN file** (it is git-ignored; `npm run check:staged` refuses it). Whether to commit the MLP file is a team decision.
3. If you want this run on record: replace `docs/MODEL_RESULTS.md` with the one from the zip, update section 6 of
   `docs/PROJECT_STATUS.md` (training date, command, commit from `RUN_INFO.json`), run `npm run check`, open a PR.
4. Choose the confidence thresholds in `src/recognition/tuning.ts` by hand from `reports/*calibration*`, with a test.
5. Then try it on a real webcam (`docs/PHASE2_CHECKLIST.md`): that number is still NOT MEASURED.